# CIFAR10 InPCA embedding grouped by `bsel`

This notebook reproduces the two interactive 3D embedding plots using the outputs from `run_dataset_pipeline.py`, but colors points by **`bsel` (method)** instead of **`m` (architecture)**.

It makes two plots:
- Train embedding (`yh`)
- Val embedding (`yvh`)

In [26]:
from __future__ import annotations
import sys
from pathlib import Path

# Make repo imports work whether you run from repo root or notebooks/
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "utils").exists() else (cwd.parent if (cwd.parent / "utils").exists() else cwd)
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import torch as th
from utils import CDICT_BSEL
from utils.plot import plotly_3d

In [27]:
# --- Config (edit these) ---
DATASET = 'TwoMoons'
INPCA_ROOT = repo_root / 'inpca_results' / DATASET
JOIN_FN = 'all_geod'
SEED = 16

print('INPCA_ROOT:', INPCA_ROOT)
print('Seed folder exists:', (INPCA_ROOT / str(SEED)).exists())


INPCA_ROOT: /home/lgreen/projects/low-dimensional-deepnets/inpca_results/TwoMoons
Seed folder exists: True


In [28]:
def _load_projection_and_index(key: str):
    key = str(key)
    fn = f"{key}_{JOIN_FN}"

    # r contains 'xp' (embedding coordinates) and 'e' (eigs)
    r_path = INPCA_ROOT / str(SEED) / f"r_{fn}_all.p"
    didx_path = INPCA_ROOT / str(SEED) / f"didx_{fn}.p"

    if not r_path.exists():
        raise FileNotFoundError(f"Missing projection file: {r_path}")
    if not didx_path.exists():
        raise FileNotFoundError(f"Missing index file: {didx_path}")

    r = th.load(str(r_path), weights_only=False)
    idx, didx = th.load(str(didx_path), weights_only=False)
    didx = didx.reset_index(drop=True)

    # Ensure bsel exists (it should if you grouped by bsel during distance computation)
    if "bsel" not in didx.columns:
        raise KeyError(f"'bsel' missing from didx columns: {list(didx.columns)}")

    # Guarantee we have m even if runs used 'model' key
    if "m" not in didx.columns and "model" in didx.columns:
        didx = didx.assign(m=didx["model"])

    return r, didx


def _make_bsel_cdict(values: pd.Series) -> dict:
    """Stable mapping: use CDICT_BSEL for known methods; deterministic fallback otherwise."""

    uniq = list(pd.Series(values).astype(str).unique())

    cdict = {k: CDICT_BSEL[k] for k in uniq if k in CDICT_BSEL}

    missing = sorted([k for k in uniq if k not in cdict])
    if missing:
        import seaborn as sns
        pal = sns.color_palette("Set2", n_colors=len(missing)).as_hex()
        for k, c in zip(missing, pal):
            cdict[k] = c

    return cdict


## Plot 1 — Train embedding (`yh`) colored by `bsel`

In [29]:
r_yh, didx_yh = _load_projection_and_index('yh')

cdict_bsel = _make_bsel_cdict(didx_yh['bsel'])

hover_cols = [c for c in ['bsel','m','seed','opt','bs','lr','wd','ratio','t','err','verr'] if c in didx_yh.columns]

fig = plotly_3d(
    didx_yh.reset_index(drop=True),
    r=r_yh,
    color='bsel',
    discrete_c=True,
    cdict=cdict_bsel,
    cols=hover_cols,
    size=4,
    opacity=0.85,
    xrange=[-3, 3],
    yrange=[-3, 3],
    zrange=[-3, 3],
)

fig.update_layout(title='Train (yh) embedding colored by bsel')

fig.show()


[ 1.30532594e+02  9.47986213e+00  2.28224070e+00 -1.07385652e+00
  1.04573678e+00 -2.18621194e-01 -1.20199223e-01 -5.55644763e-02
 -3.69445358e-02]


## Plot 2 — Val embedding (`yvh`) colored by `bsel`

In [30]:
r_yvh, didx_yvh = _load_projection_and_index('yvh')

cdict_bsel_v = _make_bsel_cdict(didx_yvh['bsel'])

hover_cols = [c for c in ['bsel','m','seed','opt','bs','lr','wd','ratio','t','err','verr'] if c in didx_yvh.columns]

fig = plotly_3d(
    didx_yvh.reset_index(drop=True),
    r=r_yvh,
    color='bsel',
    discrete_c=True,
    cdict=cdict_bsel_v,
    cols=hover_cols,
    size=4,
    opacity=0.85,
    xrange=[-3, 3],
    yrange=[-3, 3],
    zrange=[-3, 3],
)

fig.update_layout(title='Val (yvh) embedding colored by bsel')

fig.show()


[ 1.29175075e+02  9.89843398e+00  2.32339297e+00  1.21064418e+00
 -1.06856918e+00 -2.37758584e-01 -1.14862776e-01 -5.85294745e-02
 -4.77704646e-02]
